# Liv2 Compulsory Exercise — Python Walkthrough

This notebook works through all seven exercises step by step in Python, mirroring `SamletScript.R`.

**Product:** A traditional Danish life-cycle unit-link policy for a 30-year-old.

| Phase | Ages | Description |
|-------|------|-------------|
| Active | 30–70 | Pension contributions (fraction theta of labour income) into a unit-link account + term life cover (K times labour income) |
| Retirement | 70–111 | Account pays a life annuity calibrated via the calculation reserve V_tilde |

Mortality intensity mu(t) is loaded from the Danish FSA 2025 benchmark (`mortality.csv`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
np.random.seed(42)

## Parameters and Setup

| Symbol | Value | Description |
|--------|-------|-------------|
| r      | 0.02  | Risk-free rate |
| alpha  | 0.04  | Risky asset drift |
| sigma  | 0.13  | Risky asset volatility |
| m      | 70    | Retirement age |
| theta  | 0.12  | Contribution rate (fraction of labour income) |
| K      | 4     | Death benefit multiple of annual labour income |
| L(t)   | 600000 * exp(0.018*(t-30)) | Labour income at age t |

**Life-cycle investment strategy pi(t):**
- 100% risky before age 55
- Linearly decreasing from 100% to 50% between ages 55 and 70
- 50% risky throughout retirement

In [ ]:
mortality = pd.read_csv("/workspace/mortality.csv", sep=";", decimal=",")
mu_t = interp1d(
    mortality["Age"], mortality["Intensity"],
    kind="linear", bounds_error=False,
    fill_value=(mortality["Intensity"].iloc[0], mortality["Intensity"].iloc[-1])
)

r     = 0.02
alpha = 0.04
sigma = 0.13
m     = 70
theta = 0.12
K     = 4

def L(t):
    return 600000 * np.exp(0.018 * (t - 30))

def pi_t(t):
    t = np.asarray(t, dtype=float)
    return np.where(
        t < 55, 1.0,
        np.where(t <= m,
                 -(1.0 / (2*(m-55)))*t + (2*m - 55)/(2*(m-55)),
                 0.5)
    )

def r_tilde(t):
    return pi_t(t)*alpha + (1 - pi_t(t))*r

print(f"pi(30) = {pi_t(30):.2f}  -> 100% risky")
print(f"pi(55) = {pi_t(55):.2f}  -> starts decreasing")
print(f"pi(70) = {pi_t(70):.2f}  -> 50% risky at retirement")
print(f"pi(90) = {pi_t(90):.2f}  -> stays at 50% post-retirement")
print(f"L(30)  = {L(30):,.0f} DKK")
print(f"L(70)  = {L(70):,.0f} DKK")

---
## Exercise (i): Annuity Reserve V_tilde_0(t) via Thiele's ODE

The annuity reserve for a life annuity paying 1 per year satisfies Thiele's ODE:

    dV/dt = (r_tilde(t) + mu_tilde(t)) * V(t) - 1

with boundary condition **V(111) = 0** (reserve exhausted at maximum age).

- **-1**: normalised annuity payout rate.
- **(r_tilde + mu)*V**: interest credited plus mortality release (accounts of dead policyholders released to the reserve).
- Calculation basis: **mu_tilde = mu** and **r_tilde_tilde = r_tilde** (same as actual).

We solve **backward** from t=111 to t=30 using **4th-order Runge-Kutta (RK4)**.

Why backward? The boundary condition V(111)=0 is at the terminal time, not the initial time. We march backward in time from the known terminal value.

In [ ]:
def solve_thiele_rk4(t_start=30, t_end=111, n=1000):
    h = (t_start - t_end) / n   # negative: step goes backward in time
    t_vec = np.linspace(t_end, t_start, n + 1)
    V_vec = np.zeros(n + 1)
    V_vec[0] = 0.0              # boundary: V(111) = 0

    def ode(t, V):
        return (r_tilde(t) + float(mu_t(t))) * V - 1.0

    for i in range(n):
        t = t_vec[i]
        V = V_vec[i]
        k1 = h * ode(t,         V)
        k2 = h * ode(t + h/2,   V + k1/2)
        k3 = h * ode(t + h/2,   V + k2/2)
        k4 = h * ode(t + h,     V + k3)
        V_vec[i + 1] = V + (k1 + 2*k2 + 2*k3 + k4) / 6

    return interp1d(t_vec, V_vec, kind="linear", bounds_error=False, fill_value=0.0)

V_func = solve_thiele_rk4()

print(f"V_tilde_0(30)  = {V_func(30):.4f}")
print(f"V_tilde_0(70)  = {V_func(70):.4f}")
print(f"V_tilde_0(111) = {V_func(111):.6f}  (should be ~0)")

In [ ]:
t_grid = np.linspace(30, 111, 500)
fig, ax = plt.subplots()
ax.plot(t_grid, V_func(t_grid), linewidth=1.5)
ax.axvline(x=m, linestyle="--", color="gray", label="Retirement m=70")
ax.set_xlabel("Age t")
ax.set_ylabel("V_tilde_0(t)")
ax.set_title("Exercise (i): Annuity reserve V_tilde_0(t)")
ax.legend()
plt.tight_layout()
plt.show()

The reserve peaks near retirement and decreases to 0 at age 111. At age 70 you need the most capital to fund all remaining annuity payments; as you age, fewer years remain and the reserve declines.

---
## Exercise (ii): Simulate Account Path X(t)

**Active phase**, t in [30, 70):

    dX = [(r_tilde(t) + mu(t))*X + theta*L(t) - mu(t)*K*L(t)] dt + sigma*pi(t)*X dW

- `r_tilde(t)*X`: investment return
- `+ mu(t)*X`: mortality credit (accounts of dead policyholders redistributed to survivors)
- `+ theta*L(t)`: premium contributions
- `- mu(t)*K*L(t)`: expected death benefit outgo
- `sigma*pi(t)*X dW`: stochastic investment component

**Retirement phase**, t in [70, 111]:

    dX = [(r_tilde(t) + mu(t))*X - X/V_tilde_0(t)] dt + sigma*pi(t)*X dW

- `- X/V_tilde_0(t)`: annuity payout rate (calibrated to exhaust account at t=111)
- `+ mu(t)*X`: mortality credit

**Euler-Maruyama** with step h = 1/10000 (h = 1/1000 below for speed).

In [ ]:
def simulate_account_path(t_start=30, t_end=111, h=1/1000, X_init=0.0, seed=None):
    if seed is not None:
        np.random.seed(seed)
    t_seq = np.arange(t_start, t_end + h/2, h)
    N = len(t_seq)
    X = np.zeros(N)
    X[0] = X_init
    Z = np.random.normal(size=N - 1)

    for i in range(N - 1):
        t    = t_seq[i]
        x    = X[i]
        pi   = float(pi_t(t))
        mu   = float(mu_t(t))
        diff = sigma * pi * x

        if t < m:
            drift = (r_tilde(t) + mu)*x + theta*L(t) - mu*K*L(t)
        else:
            drift = (r_tilde(t) + mu)*x - x / float(V_func(t))

        X[i + 1] = x + drift*h + diff*np.sqrt(h)*Z[i]

    return t_seq, X

t_seq, X_path = simulate_account_path(h=1/1000, seed=7)

print(f"X(30)  = {np.interp(30,  t_seq, X_path):,.0f}  (should be 0)")
print(f"X(70)  = {np.interp(70,  t_seq, X_path):,.0f}  (savings pot at retirement)")
print(f"X(111) = {np.interp(111, t_seq, X_path):,.0f}  (should be ~0)")

---
## Exercise (iii): Plot and Comment

In [ ]:
fig, ax = plt.subplots()
ax.plot(t_seq, X_path / 1e6, linewidth=1.2)
ax.axvline(x=m, linestyle="--", color="gray", label="Retirement m=70")
ax.set_xlabel("Age t")
ax.set_ylabel("Account value (millions DKK)")
ax.set_title("Exercises (ii)/(iii): Simulated account path X(t)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Account value at retirement: {np.interp(m, t_seq, X_path)/1e6:.2f} million DKK")

**Comment:**

- **Starts at X(30) = 0** — no savings at policy start.
- **Accumulates** during active phase: contributions and investment returns outweigh the death benefit costs for a young policyholder.
- **Peaks at retirement (m=70)**: the savings pot that funds the annuity.
- **Decumulates** during retirement: the payout rate X/V_tilde drains the account.
- **Reaches X(111) ≈ 0**: V_tilde was calibrated to achieve this — the payout accelerates as V_tilde → 0.
- **Random fluctuations** reflect the Brownian motion driving the risky asset.

---
## Exercise (iv): Market Reserve Equals Account Value

**Claim:** V^0(t, x) = x

**Strategy:** Verification — show V^0 = x satisfies Thiele's PDE in both phases with the correct terminal boundary condition, then conclude by uniqueness.

### Thiele's PDE

By Feynman-Kac, V^0(t,x) satisfies:

    dV^0/dt + a(t,x)*dV^0/dx + (1/2)*b(t,x)^2*d^2V^0/dx^2 = (r_tilde + mu)*V^0 - c(t,x)   ... (1)

where a is the drift of X, b(t,x) = sigma*pi(t)*x is the diffusion, c(t,x) is net outflows (benefits minus premiums).

### Derivatives of V^0(t,x) = x

    dV^0/dt = 0,    dV^0/dx = 1,    d^2V^0/dx^2 = 0

The second-order term vanishes (linear in x), and the time derivative is zero. PDE (1) reduces to:

    a(t,x) = (r_tilde + mu)*x - c(t,x)

We verify this in each phase.

### Retirement phase, t in [m, 111]

- Drift from SDE (2): `a = (r_tilde + mu)*x - x/V_tilde`
- Net outflow: `c = x/V_tilde`  (annuity payout; no death benefit after retirement)

Check:

    a - (r_tilde+mu)*x + c
    = [(r_tilde+mu)*x - x/V_tilde] - (r_tilde+mu)*x + x/V_tilde
    = 0  checkmark

### Active phase, t in [30, m)

- Drift from SDE (1): `a = (r_tilde + mu)*x + theta*L - mu*K*L`
- Net outflow: `c = mu*K*L - theta*L`  (death benefit minus premium)

Check:

    a - (r_tilde+mu)*x + c
    = [(r_tilde+mu)*x + theta*L - mu*K*L] - (r_tilde+mu)*x + [mu*K*L - theta*L]
    = 0  checkmark

### Boundary condition

V^0(111, x) = 0 requires X(111) = 0. Since V_tilde(111) = 0, the payout rate x/V_tilde diverges near t=111, draining the account completely. The simulation confirms X(111) ≈ 0.

### Conclusion

V^0(t,x) = x is the unique market reserve. The policyholder bears all investment risk; the account IS the self-financing replicating portfolio for all future obligations.

---
## Exercise (v): Life Annuity Benefit Y(t) as a Generalised GBM

Define the **life annuity benefit** for t >= m:

    Y(t) = X(t) / V_tilde_0(t)

Y(t) is the annual benefit rate: the insured receives Y(t) DKK/year.

### Deriving dY

V_tilde_0(t) is **deterministic**, so the quotient rule (no second-order correction) gives:

    dY = (1/V_tilde) dX  -  (X / V_tilde^2) * (dV_tilde/dt) dt

Substitute:
1. dX from the retirement SDE: `dX = [(r_tilde + mu)*X - X/V_tilde] dt + sigma*pi*X dW`
2. dV_tilde/dt from Thiele's ODE: `dV_tilde/dt = (r_tilde_tilde + mu_tilde)*V_tilde - 1`

Expanding and collecting the dt terms (the 1/V_tilde and -1/V_tilde terms cancel):

    dY = [(r_tilde + mu)*Y - (r_tilde_tilde + mu_tilde)*Y] dt  +  sigma*pi(t)*Y dW

    dY = [(r_tilde - r_tilde_tilde) + (mu - mu_tilde)] * Y dt  +  sigma*pi(t)*Y dW

### Under the Calculation Basis (Table 2)

The basis sets `mu_tilde = mu` and `r_tilde_tilde = r_tilde`. Both differences vanish:

    dY(t) = sigma * pi(t) * Y(t) dW(t)

This is a **generalised Geometric Brownian Motion** with zero drift and time-varying volatility sigma*pi(t).

Key consequences:
- **Y(t) is a martingale:** E[Y(t)] = Y(m) for all t >= m
- The expected benefit is flat throughout retirement
- All uncertainty comes from the stochastic investment component via the Brownian motion

---
## Exercise (vi): Simulate N = 1000 Paths of Y(t)

Since drift is zero, the Euler step simplifies to:

    Y(t_{i+1}) = Y(t_i) + sigma * pi(t_i) * Y(t_i) * sqrt(h) * Z_i,    Z_i ~ N(0,1)

Initial value: Y(m) = X(m) / V_tilde_0(m), from the single simulated path in Exercise (ii).

In [ ]:
X_m = float(np.interp(m, t_seq, X_path))
Y_m = X_m / float(V_func(m))

print(f"X(70)         = {X_m:>12,.0f} DKK")
print(f"V_tilde_0(70) = {float(V_func(m)):>12.4f}")
print(f"Y(70)         = {Y_m:>12,.0f} DKK/year  (initial annual benefit)")

In [ ]:
def simulate_annuity_paths(Y_init, t_start=m, t_end=111, h=1/100, N=1000, seed=42):
    np.random.seed(seed)
    t_arr = np.arange(t_start, t_end + h/2, h)
    M = len(t_arr)
    Y_mat = np.zeros((N, M))
    Y_mat[:, 0] = Y_init
    Z = np.random.normal(size=(N, M - 1))

    for i in range(M - 1):
        t         = t_arr[i]
        pi        = float(pi_t(t))
        diff_coef = sigma * pi           # drift is zero under calculation basis
        Y_mat[:, i + 1] = (
            Y_mat[:, i]
            + diff_coef * Y_mat[:, i] * np.sqrt(h) * Z[:, i]
        )

    return t_arr, Y_mat

t_arr, Y_mat = simulate_annuity_paths(Y_init=Y_m)
print(f"Simulated {Y_mat.shape[0]} paths over {Y_mat.shape[1]} time steps")

In [ ]:
fig, ax = plt.subplots()
for j in range(15):
    ax.plot(t_arr, Y_mat[j] / 1000, alpha=0.6, linewidth=0.8)
ax.axhline(y=Y_m / 1000, linestyle="--", color="black",
           label=f"Y(70) = {Y_m/1000:.0f}k DKK/year")
ax.set_xlabel("Age t")
ax.set_ylabel("Annual benefit Y(t) (thousands DKK)")
ax.set_title("Exercise (vi): 15 sample paths of life annuity benefit Y(t)")
ax.legend()
plt.tight_layout()
plt.show()

**Comment:** Paths fan out substantially over time. Some double or more; others decline significantly. This reflects the stochastic investment risk borne entirely by the policyholder — there is no guaranteed benefit floor in a unit-link product.

---
## Exercise (vii): Expected Life Annuity Benefits

Since Y(t) is a martingale under the calculation basis:

    E[Y(t)] = Y(m)   for all t >= m

The empirical mean across N=1000 paths should stay approximately flat at Y(m).

In [ ]:
E_Y = Y_mat.mean(axis=0)

fig, ax = plt.subplots()
ax.plot(t_arr, E_Y / 1000, linewidth=1.5, label="E[Y(t)] — empirical mean (N=1000)")
ax.axhline(y=Y_m / 1000, linestyle="--", color="red",
           label=f"Y(m) = {Y_m/1000:.0f}k — theoretical expectation")
ax.set_xlabel("Age t")
ax.set_ylabel("Expected benefit (thousands DKK/year)")
ax.set_title("Exercise (vii): Expected life annuity benefit E[Y(t)]")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Y(m)      = {Y_m/1000:.2f}k")
print(f"E[Y(111)] = {E_Y[-1]/1000:.2f}k  (should be close to Y(m) by martingale property)")
print(f"Relative deviation: {abs(E_Y[-1] - Y_m)/Y_m * 100:.1f}%")

**Comment:**

- The empirical mean tracks Y(m) closely — confirming the martingale property.
- Small deviations are Monte Carlo noise (finite N=1000); they vanish as N grows.
- **Payout profile:** In expectation the insured receives the same annual benefit throughout retirement. But the realised path is highly uncertain — early good performance permanently raises the benefit; early losses permanently reduce it.
- This is the core unit-link trade-off: full upside participation at the cost of bearing all investment risk with no guaranteed floor.